# Dataset-Driven Retrieval Evaluation

**Branch:** `Klement/feat/retrieval-evaluation-dataset`
**Notebook:** `notebooks/2026-10-27-dataset-retrieval-evaluation.ipynb`

## Goal

`evaluate_retrieval()` (PR #25) scores **one** query. This feature answers a different question:

> Across a fixed set of representative questions, does Recall retrieve the right sources, and which questions fail?

You will build, end to end: a versioned evaluation dataset, a runner that calls the existing `search()`, an aggregation step, and a reproducible report. The result is a **baseline** that any later change (chunking, threshold, ranking) can be compared against.

## How to work

1. Read the part. Think before coding.
2. Write your code in the exercise cell, then move it into `src/devassistant/` once it works.
3. Use **Hint +1** only when blocked, **Hint +2** only after that.
4. Ask for a review at the end of each part. Say **bon** to validate and move on.

## Scope

| In scope | Out of scope |
|---|---|
| Dataset schema, loader, validation | Changes to `search()` or the RAG pipeline |
| Ranking extraction from search results | LLM answer evaluation (faithfulness, correctness) |
| Per-query results + macro-average | Parameter auto-tuning |
| JSON report + text summary | Charts, new dependencies |
| Single-domain evaluation | Multi-domain evaluation, persistent index |

## Plan

| Part | Topic | Exercises |
|---|---|---|
| 0 | Setup and verification | 0.1, 0.2 |
| 1 | Dataset contract and corpus | 1.1, 1.2, 1.3 |
| 2 | Ranking extraction | 2.1, 2.2 |
| 3 | Runner | 3.1, 3.2, 3.3 |
| 4 | Aggregation | 4.1, 4.2 |
| 5 | Reproducible report | 5.1, 5.2, 5.3 |
| 6 | Baseline and analysis | 6.1, 6.2, 6.3 |

## Part 0 — Setup and verification

**Why this part exists.** This notebook was written from the README, project log and architecture document, not from the source files. Every contract below that depends on existing code is marked **ASSUMPTION**. You verify them first; if reality differs, the code wins and the notebook contract is updated.

### Exercise 0.1 — Create the branch and check the baseline

```bash
git switch develop
git pull
git switch -c Klement/feat/retrieval-evaluation-dataset
python -m pytest -q
```

Record the number of passing tests. It is your regression baseline: it must not drop.

(.venv) PS C:\Users\Administrateur\Documents\Recall> python -m pytest -q
................................................................................................................ [ 99%]
.                                                                                                                [100%]
113 passed in 20.34s

In [ ]:
# Exercise 0.1 - record what you observed
baseline_tests_passed = None   # fill in from your own run
python_version = None          # fill in from `python --version`

### Exercise 0.2 — Verify the assumptions

| # | ASSUMPTION | Where to verify |
|---|---|---|
| A1 | `evaluate_retrieval()` evaluates one query and takes retrieved IDs, relevant IDs and `k` | `src/devassistant/evaluation.py` |
| A2 | It returns Precision@k, Recall@k and Reciprocal Rank@k (container type and key names unknown) | same file + `tests/test_retrieval_evaluation.py` |
| A3 | `search(documents, question, top_k=3, score_threshold=0.0)` returns a list of dicts with `chunk_score`, `source_score`, `chunk` | `src/devassistant/search.py` |
| A4 | The source name lives inside `chunk` under some key | `chunking.py` / `document_loader.py` |
| A5 | Results come back grouped or ordered by `source_score`, ties broken by source name | `search.py` + its tests |
| A6 | Documents passed to `search()` are the chunked output of the loader | `rag.py` (`retrieve_documents()`) |

Do not guess: open the files, then fill the cell below.

In [2]:
import inspect
from devassistant.evaluation import evaluate_retrieval
from devassistant.search import search
print(inspect.signature(evaluate_retrieval))
print(inspect.signature(search))

verified = {
    "A1_evaluate_retrieval_signature": True,
    "A2_return_type_and_keys": True,
    "A3_search_result_shape": True,
    "A4_source_key_in_chunk": True,
    "A5_result_order_guarantee": True,
    "A6_how_rag_builds_documents": True,
}
verified

(retrieved_ids: list[str], relevant_ids: set[str], k: int = 5) -> dict[str, float]
(documents, question, top_k=3, score_threshold=0.0)


{'A1_evaluate_retrieval_signature': True,
 'A2_return_type_and_keys': True,
 'A3_search_result_shape': True,
 'A4_source_key_in_chunk': True,
 'A5_result_order_guarantee': True,
 'A6_how_rag_builds_documents': True}

<details><summary><b>Hint +1</b></summary>

Print the signature of each function and read the docstring. Then open the existing test file: tests document the real contract better than prose.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

For A5, find the search test that uses equal scores for two sources. If none exists, that is a finding: note it, because Exercise 2.2 depends on it.

</details>

**Checkpoint 0.** You can state, without notes, what `search()` takes and returns and what `evaluate_retrieval()` takes and returns. Share your `verified` dict for review before Part 1.

## Part 1 — Dataset contract and corpus

### Concept

A retrieval dataset is a set of **(question, relevant source IDs)** pairs. The relevance labels are the ground truth: they must be written by a human, **before** looking at what Recall retrieves. Labels derived from retrieval output would make the metrics circular.

Design constraints:

- Versioned in the repository, so a baseline can be reproduced.
- Based on a **frozen corpus** dedicated to evaluation, separate from your personal `documents/` folder. If the corpus changes, old scores are no longer comparable.
- Global run parameters (`k`, `top_k`, `score_threshold`, chunking) live in the runner configuration, not in the dataset, so one dataset can be run under many configurations.

### Exercise 1.1 — Define the data contract

Decide and document:

1. File format (JSON is the proposal; confirm or argue otherwise).
2. Fields per entry. Minimum: a question and a list of relevant source IDs. Do you need a stable `id` per question to compare runs? Argue it.
3. What a source ID is: the filename only, or a path relative to the corpus directory? (Think about duplicates in subfolders.)
4. How the dataset points to its corpus.

Write the contract as a dataclass (or `TypedDict`) in the cell below, plus a 5 to 10 line docstring stating the invariants.

**Answers**
1. We can start with JSON for the moment and open to more later. Or we will create a converter *.* to *.json.
2. i don't know
3. we have discussed about the issue to have doubloons in the corpus, an id should be related to a "path+filename.*". But we can discuss again on how to manage a doubloon of a document name.
4. i don't know

In [ ]:
# Exercise 1.1
from dataclasses import dataclass

@dataclass(frozen=True)
class EvaluationCase:
    """Represent one retrieval evaluation case.

    Invariants:
        - id_eval is non-empty and contains only ASCII letters, digits,
          hyphens, or underscores.
        - question_eval contains at least one non-whitespace character.
          Questions may contain code or SQL and must be treated as text.
        - relevant_ids contains at least one source ID.
        - Source IDs are non-empty, unique within this case, and represent
          paths relative to the corpus directory.

    These invariants are documented here but are not enforced by this class.
    """

    id_eval: str
    question_eval: str
    relevant_ids: tuple[str, ...]


@dataclass(frozen=True)
class EvaluationDataset:
    """Represent evaluation cases associated with a reference corpus.

    Invariants:
        - path_corpus is non-empty and relative to the dataset JSON directory.
        - The resolved corpus path points to an existing directory.
        - eval_cases contains at least one EvaluationCase.
        - Case IDs are unique across the dataset.
        - Every relevant source ID refers to an existing file in the corpus.

    These invariants are documented here but are not enforced by this class.
    """

    path_corpus: str
    eval_cases: tuple[EvaluationCase, ...]

<details><summary><b>Hint +1</b></summary>

List the invariants first (non-empty question, at least one relevant ID, no duplicate IDs inside one case, unique question IDs). A frozen dataclass makes immutability part of the contract.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

Store `relevant_ids` as a tuple rather than a list in a frozen dataclass: a list inside a frozen dataclass is still mutable. Decide whether order of relevant IDs matters (it should not).

</details>

### Exercise 1.2 — Build the corpus and the questions

Create a small frozen corpus (suggestion: `evaluation/corpus/`, 8 to 12 short Markdown files on distinct topics) and 10 to 15 questions with labels.

Rules:

- Write each question **from a document you chose**, then label it. Do not run Recall to decide labels.
- Include variety: questions answered by exactly one source, questions answered by two sources, and at least two **hard** ones (paraphrase with little shared vocabulary, or a topic touching several files).
- Avoid questions with **no** relevant source. Recall@k is undefined there; they are out of scope for this version.
- Write everything in English (repository convention).

Deliverables: the corpus directory and `evaluation/datasets/baseline.json`.

**Reflection (answer in the cell):** with only 10 to 15 questions, what can you conclude from an average Recall@3 of 0.8 versus 0.7?

In [ ]:
# Exercise 1.2 - reflection
reflection_small_dataset = """
"""

<details><summary><b>Hint +1</b></summary>

Start from the documents, not from the questions: pick a document, ask what a developer would genuinely want to know from it, then phrase that as a question.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

For hard questions, rewrite a natural question without reusing the document's key terms. These are the cases where embeddings help or fail, and where your baseline becomes informative.

</details>

### Exercise 1.3 — Loader with validation

Implement `load_dataset(path) -> EvaluationDataset` in a new module (proposal: `src/devassistant/evaluation_dataset.py`; confirm the module split before creating it).

Required failures (each with a clear, specific error message):

- file missing, or invalid JSON;
- wrong top-level shape or missing/ill-typed fields;
- empty question, empty `relevant_ids`, duplicate IDs inside one case;
- duplicate question IDs across the dataset;
- optionally: relevant ID not present in the corpus directory.

Decision to make: which exception type(s)? One custom `DatasetError` or plain `ValueError`? Defend the choice.

In [ ]:
# Exercise 1.3
from pathlib import Path

def load_dataset(path):
    raise NotImplementedError

<details><summary><b>Hint +1</b></summary>

Separate parsing (JSON to Python objects) from validation (invariants). Two small functions are easier to test than one large one.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

Collect the context in the error message: which entry index, which field. A message like `case 7: relevant_ids contains duplicate 'rag.md'` is worth the extra line.

</details>

**Checkpoint 1.** Review request: contract (1.1), dataset file (1.2), loader (1.3). Questions I will ask you: why is the ID the filename and not the path? Why are `k` and `score_threshold` absent from the dataset?

## Part 2 — Ranking extraction

### Concept

`evaluate_retrieval()` expects a **ranked list of unique source IDs**. `search()` returns **chunks**, and several chunks may belong to one source (`top_k` selects sources, not chunks). Feeding chunk-level results to the metric would count one source several times and silently distort Precision@k.

Cutoff semantics matter: deduplicate **before** applying the cutoff (as `evaluate_retrieval()` does), so ten chunks of one source never occupy ten ranks.

### Exercise 2.1 — Extract unique sources in rank order

Implement `extract_ranked_source_ids(results) -> list[str]`:

- input: the list returned by `search()`;
- output: unique source IDs, **first occurrence order**, using the source key you verified in A4;
- empty input gives an empty list;
- it must not modify its input.

Decide: does this function trust the incoming order (A5), or does it re-sort by `source_score` then source name? State which, and why.

In [ ]:
# Exercise 2.1
def extract_ranked_source_ids(results):
    raise NotImplementedError

<details><summary><b>Hint +1</b></summary>

A `dict` preserves insertion order in Python 3.7+, which gives you ordered deduplication in one line without a set plus a list.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

If you choose to re-sort, build one `{source: source_score}` mapping first, then sort by `(-score, source)`. If you trust the order, say so in the docstring and cover it with the test of Exercise 2.2.

</details>

### Exercise 2.2 — Test it with hand-built results

You do not need embeddings here. Build `search()`-shaped dictionaries by hand and assert behavior for:

1. several chunks from one source (single rank, not several);
2. two sources with different scores in non-sorted input order;
3. two sources with **equal** scores (deterministic tie-break by name);
4. empty input;
5. input not mutated.

Write these as pytest functions in `tests/test_retrieval_evaluation_dataset.py` (names and docstrings in English). Case 2 and 3 will tell you whether Exercise 2.1 should trust or re-sort.

In [ ]:
# Exercise 2.2 - sketch your fixtures here
def make_result(source, chunk_score, source_score):
    raise NotImplementedError

<details><summary><b>Hint +1</b></summary>

A tiny factory `make_result(source, chunk_score, source_score)` keeps the tests readable. Use the real chunk shape you verified in A3/A4, not an invented one.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

If the tie-break test passes only because of dictionary order in your fixture, change the fixture order and run again: a test that cannot fail proves nothing.

</details>

**Checkpoint 2.** Review request. Question I will ask: what happens to Precision@3 if you skip deduplication and `search()` returns 3 chunks from the same source?

## Part 3 — Runner

### Concept

The runner glues the pieces together for one query, then for the whole dataset. To keep unit tests free of LM Studio, the search step must be **injectable**: the runner receives a callable instead of importing and calling the real search directly. This is plain dependency injection, the same idea as passing an interface to a C++ class instead of hard-wiring a singleton.

### Exercise 3.1 — Configuration object

Define a frozen `EvaluationConfig` holding everything that characterizes a run: `k`, `top_k`, `score_threshold`, `chunk_size`, `overlap`, corpus directory. It will be written into the report so a run can be reproduced.

**Decision (write the justification):** what is the relationship between `k` and `top_k`? `top_k` is the number of selected **sources**; `k` is the metric cutoff. If `k > top_k`, the retrieved list can never reach `k` entries and Precision@k is penalized mechanically. Options: reject `k > top_k` at construction, default `top_k = k`, or allow it and document the effect. Pick one.

In [ ]:
# Exercise 3.1
# @dataclass(frozen=True)
# class EvaluationConfig:
#     ...

<details><summary><b>Hint +1</b></summary>

Validation belongs in `__post_init__`. Fail early with a message naming both values.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

Rejecting `k > top_k` is the strictest choice and the simplest to explain at an oral. Allowing it requires you to explain a metric artifact every time you present a number.

</details>

### Exercise 3.2 — Run one query

Implement `run_query(case, documents, config, search_fn) -> QueryResult`.

`QueryResult` must keep, per query: the case ID, the question, the ranked retrieved source IDs (as passed to the metric), the relevant IDs, and the three metric values. Keeping the retrieved list is what lets you diagnose a failing query later.

Constraints:

- `search_fn(documents, question, top_k=..., score_threshold=...)` has the same signature as the real `search()` (verify against A3).
- No LM Studio call inside this function.
- Use `evaluate_retrieval()` as-is. Do not reimplement the metrics.

In [ ]:
# Exercise 3.2
def run_query(case, documents, config, search_fn):
    raise NotImplementedError

<details><summary><b>Hint +1</b></summary>

Write the test first with a fake `search_fn` returning fixed results. If the test is hard to write, the function has too many responsibilities.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

Pipeline of four lines: call search_fn, extract ranked IDs (Part 2), call evaluate_retrieval with `config.k`, build the QueryResult.

</details>

### Exercise 3.3 — Run the dataset

Implement `run_dataset(dataset, config, search_fn, load_documents_fn) -> list[QueryResult]`.

Design points:

- Documents are loaded and chunked **once**, not once per question.
- The order of results follows the dataset order (deterministic).
- Decide the failure policy: if one query raises (for example an LM Studio timeout), does the whole run abort, or is the error recorded in that query's result? For a baseline, silent skipping is the worst option because it biases the average. Defend your choice.

In [ ]:
# Exercise 3.3
def run_dataset(dataset, config, search_fn, load_documents_fn):
    raise NotImplementedError

<details><summary><b>Hint +1</b></summary>

Injecting `load_documents_fn` as well lets you test that loading happens once (count the calls in a fake).

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

Fail-fast is the simplest honest policy for a first version: any error aborts the run with the case ID in the message. Recording errors per query is better later, but then the aggregation must define how errors are counted.

</details>

**Checkpoint 3.** Review request. Questions I will ask: how do your tests prove `run_dataset` does not call the real search? Where do embedding-cache effects show up in a baseline run, and does the cache change the metrics or only the latency?

## Part 4 — Aggregation

### Concept

Macro-average: compute each metric per query, then average across queries, giving each query equal weight. This differs from micro-averaging (pooling all retrieved items), which would let queries with many relevant items dominate.

An average alone hides failures: a mean Recall@3 of 0.75 can be fifteen perfect queries and five total misses. Always keep the per-query list next to the mean.

### Exercise 4.1 — Macro-average

Implement `aggregate(results) -> AggregateMetrics` returning the mean of Precision@k, Recall@k and Reciprocal Rank@k (the mean of Reciprocal Rank is the standard **MRR**).

Define explicitly:

- behavior on an empty list (raise, or return zeros? zeros would look like a real score);
- the number of queries included in the aggregate;
- floating-point comparison in tests (`pytest.approx`).

In [ ]:
# Exercise 4.1
def aggregate(results):
    raise NotImplementedError

<details><summary><b>Hint +1</b></summary>

An empty dataset has no meaningful mean. Raising a `ValueError` is more honest than returning 0.0.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

Compute the mean of each field with `sum(...) / len(results)` over the per-query values, in a single pass over `results` if you want to keep it simple.

</details>

### Exercise 4.2 — Verify by hand

Build three `QueryResult` objects with metric values you can compute mentally (for example Precision 1.0 / 0.5 / 0.0), predict the aggregate on paper, then assert it. Add one case that demonstrates why the mean hides a failure, and write one sentence about it in the cell below.

In [ ]:
# Exercise 4.2 - one sentence
mean_hides_failures = """
"""

<details><summary><b>Hint +1</b></summary>

Choose values whose mean is exact in binary floating point (0.5, 0.25, 1.0) so you can debug the test without tolerance doubts.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

The failure demo: two queries at 1.0 and one at 0.0 gives a mean of about 0.667 while one query got nothing.

</details>

**Checkpoint 4.** Review request. Question I will ask: why macro and not micro for this dataset?

## Part 5 — Reproducible report

### Concept

A number is only useful if you can tell **what produced it**. The report must contain the configuration, the dataset identity, per-query results and the aggregate. Two runs with the same inputs must give the same report.

Reproducibility pitfalls to handle: timestamps (keep them out of the compared part or put them in a separate field), float formatting, dictionary key order, absolute paths.

### Exercise 5.1 — Report structure and serialization

Define a `EvaluationReport` (config, dataset name/path, per-query results, aggregate) and `to_dict()` / `write_json(path)`.

Decide: how do you identify the dataset version? A hash of the file content is cheap and removes ambiguity ("which baseline.json was that?"). Is it worth it here? Argue.

In [ ]:
# Exercise 5.1
# @dataclass(frozen=True)
# class EvaluationReport:
#     ...

<details><summary><b>Hint +1</b></summary>

`json.dumps(..., indent=2, sort_keys=True)` gives stable output. Test it by serializing twice and comparing strings.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

`hashlib.sha256(path.read_bytes()).hexdigest()` identifies the dataset file. Store a short prefix in the report; a full hash is noise for a human reader.

</details>

### Exercise 5.2 — Text summary

Implement `format_summary(report) -> str`: configuration line, the three aggregate metrics, and a table of per-query results with the **worst queries first** (lowest Recall@k, then lowest Reciprocal Rank). A reader should see failures without scrolling.

No new dependency: plain string formatting.

In [ ]:
# Exercise 5.2
def format_summary(report):
    raise NotImplementedError

<details><summary><b>Hint +1</b></summary>

Sort a copy of the per-query list; do not reorder the report's stored results, because stored order must remain dataset order.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

Use f-string width specifiers (`{value:.3f}`, `{text:<40}`) and truncate long questions so columns stay aligned.

</details>

### Exercise 5.3 — Entry point

Provide a way to run the evaluation from the command line, for example `python -m devassistant.evaluation_runner --dataset ... --k 3 --output ...` (module name to confirm).

Constraints:

- importing the module must not trigger a run or an LM Studio call;
- an LM Studio failure must produce a readable message and a non-zero exit code;
- the default output path must not overwrite a previous baseline silently.

Use `argparse` from the standard library.

In [ ]:
# Exercise 5.3
def main(argv=None):
    raise NotImplementedError

# if __name__ == "__main__":
#     raise SystemExit(main())

<details><summary><b>Hint +1</b></summary>

Make `main` take `argv` so a test can call `main([...])` without spawning a process.

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

Inject the real `search` and loader only inside `main`; everything below it stays injectable and unit-tested.

</details>

**Checkpoint 5.** Review request. Question I will ask: how do you prove two runs produce an identical report?

## Part 6 — Baseline and analysis

### Exercise 6.1 — Run the real baseline

With LM Studio running and the embedding model loaded, run the evaluation on `baseline.json` with a documented configuration. Save the JSON report in the repository (suggestion: `evaluation/reports/`), with the date in the file name.

Record the exact command, the configuration, and the three aggregate values. Mark clearly that these numbers come from **your** run: nobody else's numbers belong in the project log.

In [ ]:
# Exercise 6.1 - record what you observed
baseline_command = None
baseline_precision_at_k = None
baseline_recall_at_k = None
baseline_mrr = None

### Exercise 6.2 — Read the failures

Take the three worst queries. For each, look at the retrieved list, the expected sources, and the chunks involved, then write a **hypothesis** (not a conclusion) among, for instance: vocabulary mismatch between question and document, chunk boundaries cutting the relevant passage, threshold excluding the right chunk, a label that is itself questionable.

Each hypothesis needs a proposed experiment that changes **one** parameter. Do not run the experiments in this feature: the output of this exercise is the list that justifies the next feature.

In [ ]:
# Exercise 6.2
failure_analysis = [
    # {"case_id": ..., "retrieved": [...], "expected": [...], "hypothesis": "...", "experiment": "..."},
]

<details><summary><b>Hint +1</b></summary>

Look first at whether the expected source appears anywhere in the ranking (rank 4 is a different problem from absent).

</details>

<details><summary><b>Hint +2</b> (open only after trying +1)</summary>

If the expected source is absent, rerun that single query with `score_threshold=-1.0` and a larger `top_k` in a scratch cell to see whether the threshold or the embedding is responsible. Do not commit that change.

</details>

### Exercise 6.3 — Documentation and wrap-up

Update, in English:

- `README.md`: usage of the runner, dataset format, corrected test count and Python version, the evaluation exercise no longer marked in progress;
- `PROJECT_LOG.md`: a dated entry with the PR, the **observed** test result and baseline values, and the known limitations (dataset size, single domain, no no-answer questions);
- `docs/architecture/system-architecture.md`: components table and evaluation section;
- the previous notebook if it still presents the exercise as unimplemented.

Then run the full test suite, review your own diff, and open the pull request (title, summary, tests, limitations).

**Oral defense questions (answer in your own words, no notes):**

1. Why a frozen corpus and human labels instead of labels generated from retrieval output?
2. What does Recall@k measure that Precision@k does not, and vice versa?
3. Why can this evaluation say nothing about answer quality?
4. With this dataset size, what difference between two configurations would you refuse to call significant?
5. What would you change first, and what evidence from your failure analysis supports it?

## Definition of done

- [ ] All exercises completed and moved into `src/devassistant/`.
- [ ] Dataset, corpus and baseline report versioned in the repository.
- [ ] Unit tests use injected fakes; no test needs LM Studio.
- [ ] Full test suite run, count recorded from your own execution.
- [ ] Real baseline run recorded with command and configuration.
- [ ] README, project log and architecture document updated.
- [ ] Pull request opened with limitations stated.